# 부록 C. 비밀정보 관리 — 키를 지키는 법 · Appendix C — Handling Secrets: Keeping Your Keys Safe

전산실험 2 · AI Agent 만들기 · **2주차** · 예상 소요 20분
Computational Experiments 2 · Building AI Agents · **Week 2** · about 20 minutes

## 목표 · Objectives
- 키를 커밋하면 실제로 무슨 일이 벌어지는지 안다
- **파일에서 지워도 이력에는 남는다**는 것을 직접 확인한다
- `.gitignore` / Colab Secrets / 환경변수를 구분해 쓴다
- 유출 사고 시의 **대처 순서**를 안다

> **EN**
> - Know what actually happens when you commit a key
> - See for yourself that **deleting it from the file leaves it in the history**
> - Use `.gitignore`, Colab Secrets and environment variables for the right jobs
> - Know **the order of response** when a leak happens

> **이 노트북은 2주차 부록입니다.** 본 진도(예제 01~03: 토큰·컨텍스트·비용)와
> 별개로, 앞으로 12주 동안 쓸 작업 환경을 갖추는 인프라 실습입니다.
> 오늘 못 끝내도 됩니다. 다만 **4주차 전에는 반드시 마쳐 두십시오.**
>
> **EN — this notebook is a week-2 appendix.** Separate from the main track, it
> sets up the working environment for the next twelve weeks. You need not finish
> it today, but **you must finish it before week 4.**

---
## 1. 실제로 일어나는 일 · What actually happens

```
09:00  api_key = "sk-abc123..." 를 커밋하고 push
09:12  GitHub를 훑는 봇이 키를 발견
09:15  키가 도용되기 시작
10:00  파일에서 지우고 다시 commit + push
       ★ 소용 없음 - 과거 커밋에 그대로 남아 있다 ★

09:00  commit and push `api_key = "sk-abc123..."`
09:12  a bot crawling GitHub finds the key
09:15  the key starts being abused
10:00  delete it from the file, commit and push again
       ★ no use — it is still sitting in the earlier commit ★
```

겁주려는 이야기가 아니라 매일 벌어지는 일입니다.
공개 저장소를 훑으며 API 키 패턴을 찾는 봇이 상시로 돌아다닙니다.

**직접 확인해 봅시다.**

> **EN** — This is not scaremongering; it happens every day. Bots crawl public
> repositories for API-key patterns around the clock. **Let's confirm it
> ourselves.**

In [ ]:
!rm -rf /content/leak_demo
!mkdir -p /content/leak_demo
%cd /content/leak_demo
!git init -q
!git config user.name "test"
!git config user.email "test@example.com"

# 실수: 키를 코드에 적었다 / The mistake: the key was written into the code
with open("agent.py", "w", encoding="utf-8") as f:
    f.write('API_KEY = "sk-SUPER-SECRET-12345"\n')
    f.write('print("에이전트 시작")\n')

!git add agent.py
!git commit -q -m "에이전트 초기 버전"
!cat agent.py

In [ ]:
# 아차 싶어서 지웠다 / We catch the mistake and remove it
with open("agent.py", "w", encoding="utf-8") as f:
    f.write('import os\n')
    f.write('API_KEY = os.environ["KMU_API_KEY"]\n')
    f.write('print("에이전트 시작")\n')

!git add agent.py
!git commit -q -m "키를 환경변수로 변경"

print("=== 현재 파일 ===")
!cat agent.py

In [ ]:
# 그런데... 이력을 뒤져 보면?
# EN — But what if we search the history?
print("=== 저장소 전체 이력에서 키 검색 ===")
!git log -p --all | grep -n "SUPER-SECRET" || echo "(없음)"

print()
print("=== 첫 커밋의 파일 내용 ===")
!git show HEAD~1:agent.py

> **키가 그대로 남아 있습니다.**
>
> Git은 모든 이력을 보관하는 도구입니다. 그것이 장점이자, 이 경우에는 함정입니다.
> 저장소를 clone 한 사람은 누구나 과거 커밋을 열어 볼 수 있습니다.

> **EN — the key is still there.** Git is a tool that preserves all history —
> its great strength, and in this case the trap. Anyone who clones the repository
> can open any past commit.

---
## 2. 유출되었을 때의 대처 순서 · The order of response after a leak

**순서가 중요합니다.**

| 순위 | 할 일 | 왜 |
|---|---|---|
| **1** | **키를 폐기(revoke)하고 새로 발급** | 이미 유출된 키는 지워도 소용없다. **무효화가 유일한 해결** |
| 2 | 코드에서 제거하고 환경변수로 교체 | 재발 방지 |
| 3 | 사용 이력 확인 | 도용 여부 점검 |
| 4 | (필요시) 이력 정리 | `git filter-repo` 등. 어렵고 위험하다 |

**EN — the order matters.**

| Rank | Do this | Why |
|---|---|---|
| **1** | **Revoke the key and issue a new one** | Deleting a leaked key achieves nothing; **invalidation is the only fix** |
| 2 | Remove it from the code, switch to an env var | Prevent a repeat |
| 3 | Review the usage log | Check for abuse |
| 4 | (If needed) rewrite the history | `git filter-repo` and friends — hard and risky |

> **기억할 것: 지우는 것이 먼저가 아니라 폐기가 먼저입니다.**
> **EN — remember: revoke first, delete second.**

이 수업에서 유출 대상이 될 만한 것들:

| 값 | 조치 |
|---|---|
| `GITHUB_TOKEN` | GitHub → Settings → Developer settings → 해당 토큰 **Delete** |
| `GEMINI_API_KEY` | Google AI Studio → API keys → **Delete** |
| `KMU_API_KEY` | **교수에게 즉시 알릴 것** (공용 키이므로 전체에 영향) |
| `KMU_BASE_URL` | **교수에게 즉시 알릴 것.** 주소는 폐기할 수 없으므로 더 곤란하다 |

**EN** — What can leak in this course:

| Value | Action |
|---|---|
| `GITHUB_TOKEN` | GitHub → Settings → Developer settings → **Delete** that token |
| `GEMINI_API_KEY` | Google AI Studio → API keys → **Delete** |
| `KMU_API_KEY` | **Tell the instructor at once** — it is shared, so everyone is affected |
| `KMU_BASE_URL` | **Tell the instructor at once.** Worse, because a URL cannot be revoked |

---
## 3. 올바른 방법 세 가지 · Three ways to do it right

### 방법 A. Colab Secrets — 이 수업의 기본 · Option A — Colab Secrets, our default
왼쪽 **열쇠 아이콘(🔑)** → 이름/값 저장 → 노트북별 액세스 토글

- 노트북 파일에 값이 들어가지 않는다
- 노트북을 공유해도 키는 따라가지 않는다
- 여러 노트북에서 재사용된다

> **EN** — Key icon (🔑) on the left → store a name and value → toggle access per
> notebook. The value never enters the notebook file, sharing the notebook does
> not share the key, and it is reusable across notebooks.

### 방법 B. 환경변수 — 서버·자동실행 환경 · Option B — environment variables, for servers and automation
### 방법 C. `.env` 파일 + `.gitignore` — 로컬 개발 · Option C — a `.env` file plus `.gitignore`, for local development

In [ ]:
# 방법 A: Colab Secrets (권장) / Option A: Colab Secrets (recommended)
def get_key(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            print(f"{name}: Secrets 에서 가져옴")
            return v.strip()
    except Exception as exc:
        print(f"(Secrets 사용 불가: {type(exc).__name__})")

    import os
    if os.environ.get(name):
        print(f"{name}: 환경변수에서 가져옴")
        return os.environ[name].strip()

    import getpass
    return getpass.getpass(f"{name}: ").strip()


키 = get_key("KMU_API_KEY")
print("길이:", len(키), "글자")

# 절대 하지 말 것:  print(키)
# EN — Never do this:  print(키)

In [ ]:
# 방법 C: .env 파일 (Colab 밖에서 개발할 때)
# EN — Option C: a .env file, for development outside Colab
%cd /content/leak_demo

with open(".env", "w", encoding="utf-8") as f:
    f.write("KMU_API_KEY=sk-실제키가여기에\n")
    f.write("GEMINI_API_KEY=AIza실제키가여기에\n")

# 반드시 .gitignore 에 넣는다 -- 파일을 만들기 '전에' 하는 것이 안전
# EN — It MUST go in .gitignore, and writing that BEFORE the file is the safe order.
with open(".gitignore", "w", encoding="utf-8") as f:
    f.write(".env\n*.key\n")

!git status --short
print("---")
print(".env 가 목록에 없으면 성공입니다.")

In [ ]:
# .env 읽어서 환경변수로 올리기 / Read .env into the environment
import os

def load_env(path=".env"):
    if not os.path.exists(path):
        print(f"{path} 없음")
        return
    for line in open(path, encoding="utf-8"):
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip()
    print(f"{path} 를 환경변수로 읽었습니다.")

load_env()
print("등록된 키:", [k for k in os.environ if k.endswith("_API_KEY")])

---
## 4. 실수 방지 체크리스트 · A checklist against accidents

커밋하기 전에 습관적으로 확인하십시오.

> **EN** — Make checking this a habit before every commit.

In [ ]:
# 커밋 전 자가 점검 - 스테이징된 내용에 키 패턴이 있는가?
# EN — Pre-commit self-check: does the staged diff contain a key pattern?
import subprocess, re

패턴들 = {
    "OpenAI 형식":  r"sk-[A-Za-z0-9_-]{20,}",
    "Google 형식":  r"AIza[A-Za-z0-9_-]{30,}",
    "GitHub PAT":   r"gh[pousr]_[A-Za-z0-9]{30,}",
    "의심 변수명":   r"(?i)(api_?key|token|password|secret)\s*=\s*[\"'][^\"']{8,}[\"']",
}

diff = subprocess.run(["git", "diff", "--cached"],
                      capture_output=True, text=True).stdout

발견 = False
for 이름, 패턴 in 패턴들.items():
    for m in re.finditer(패턴, diff):
        발견 = True
        표시 = m.group()[:12] + "..."      # 전체를 출력하지 않는다 / never print the whole match
        print(f"[경고] {이름}: {표시}")

print("의심스러운 것 없음" if not 발견 else "\n>>> 커밋을 중단하고 확인하세요!")

> 이 셀을 자신의 저장소 `src/precommit_check.py` 로 저장해 두고
> 커밋 전에 한 번씩 돌려 보십시오.
>
> GitHub에는 **Secret scanning** 기능이 있어 공개 저장소의 키를 자동 탐지하고
> 알림을 보내 줍니다. 하지만 이미 push 된 뒤입니다. 마지막 방어선일 뿐입니다.

> **EN** — Save this cell as `src/precommit_check.py` in your own repository and
> run it before each commit.
>
> GitHub has **secret scanning**, which detects keys in public repositories and
> alerts you — but only after the push. It is the last line of defence, not the
> first.

---
## 5. 설정과 비밀정보를 분리하기 · Separating configuration from secrets

모든 설정이 비밀은 아닙니다. 섞지 마십시오.

| 종류 | 예 | 어디에 |
|---|---|---|
| **비밀** | API 키, 토큰, 비밀번호 | Secrets / 환경변수. **절대 커밋 금지** |
| 공개 설정 | 서버 주소, 모델 이름, 온도 기본값 | 코드나 설정 파일에. **커밋해도 됨** |

**EN** — Not every setting is a secret. Do not mix them.

| Kind | Examples | Where it lives |
|---|---|---|
| **Secret** | API keys, tokens, passwords | Secrets / env vars. **Never commit** |
| Public config | Server URL, model name, default temperature | Code or a config file. **Safe to commit** |

`kmu_llm.py` 를 다시 보면 이 원칙이 지켜져 있습니다.
`PROVIDERS` 에는 주소와 모델 이름만 있고, 키는 `get_key()` 로 **실행 시점에** 가져옵니다.

> **EN** — Look back at `kmu_llm.py` and you will see the principle upheld:
> `PROVIDERS` holds only URLs and model names, while the key is fetched **at run
> time** by `get_key()`.
>
> (One nuance for this course: the department URL is treated as a secret too,
> because the repository is public.)

In [ ]:
# 공개 설정은 파일로 관리해도 좋다 / Public configuration can live in a file
import json

config = {
    "provider": "kmu",
    "temperature": 0.0,
    "max_tokens": 1024,
    "max_agent_steps": 10,
}

with open("config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

!cat config.json
print()
print("이 파일은 커밋해도 안전합니다. 키가 없으니까요.")

---
## 연습문제 · Exercises

**문제 1.** 실수로 키를 커밋했다고 가정하시오.
무엇을 어떤 순서로 할 것인지 4단계로 적으시오.

**문제 2.** 위 자가 점검 코드를 `src/precommit_check.py` 로 저장하고
자신의 저장소에 커밋하시오.

**문제 3.** `.gitignore` 에 `.env` 를 넣기 **전에** 이미 `.env` 를 커밋했다면,
지금 `.gitignore` 를 추가하는 것으로 문제가 해결되는가? 직접 실험해서 확인하시오.

**문제 4.** 이 수업의 `KMU_API_KEY` 는 모든 수강생이 공유합니다.
한 사람이 이것을 공개 저장소에 올리면 어떤 일이 생기는가?
개인별 키를 쓰는 방식과 비교해 장단점을 적으시오.

> **EN — 1.** Suppose you committed a key by mistake. Write down what you do, in
> four ordered steps.
>
> **EN — 2.** Save the self-check code above as `src/precommit_check.py` and
> commit it to your repository.
>
> **EN — 3.** If `.env` was already committed **before** you added it to
> `.gitignore`, does adding it now solve the problem? Try it and find out.
>
> **EN — 4.** The `KMU_API_KEY` in this course is shared by every student. What
> happens if one person pushes it to a public repository? Write down the pros and
> cons against per-student keys.

In [ ]:
# 여기에 직접 작성해 보세요 / Write your own code here

<details>
<summary><b>해설 보기 · Show the solutions</b></summary>

**문제 1.**
1. **키 폐기 후 재발급** (가장 먼저!)
2. 코드에서 제거 → 환경변수/Secrets 로 교체 → 커밋
3. 사용 이력·과금 확인
4. 필요하면 `git filter-repo` 로 이력 정리 (공동 저장소라면 팀에 먼저 알릴 것)

**문제 3.**
해결되지 않습니다. `.gitignore` 는 **아직 추적되지 않는 파일**에만 적용됩니다.
이미 추적 중이면 계속 추적됩니다.
```
!git rm --cached .env
!git commit -m "추적 대상에서 .env 제외"
```
그래도 **과거 커밋에는 남아 있습니다.** 결국 답은 키 폐기입니다.

**문제 4.**
공용 키가 유출되면 **전원이 영향을 받습니다.** 키를 교체하면 모두가 Secrets를
다시 설정해야 합니다. 도용이 일어나도 누가 유출했는지 추적할 수 없습니다.

| | 공용 키 | 개인 키 |
|---|---|---|
| 배포 | 쉬움 (수업 중 한 번) | 번거로움 (개별 발급) |
| 유출 영향 | 전원 | 본인만 |
| 사용량 추적 | 불가 | 가능 |
| 폐기 비용 | 전원 재설정 | 본인만 |

실제 서비스라면 반드시 개인별 키에 사용량 제한을 겁니다.
수업에서는 편의를 위해 공용 키를 쓰는 것이니, **그만큼 조심**하십시오.

---

**EN — 1.** (1) **Revoke and reissue the key** — first, always. (2) Remove it
from the code, switch to an env var or secret, commit. (3) Review the usage and
billing records. (4) If necessary, rewrite the history with `git filter-repo`
(tell your collaborators first if the repository is shared).

**EN — 3.** No. `.gitignore` applies only to **files not yet tracked**; an
already-tracked file stays tracked, and removing it takes the explicit
`git rm --cached` shown above. Even then **it remains in the past commits**. The
answer, in the end, is to revoke the key.

**EN — 4.** When a shared key leaks, **everyone is affected**: replacing it means
every student must reconfigure Secrets, and if it is abused there is no way to
trace who leaked it.

| | Shared key | Per-student key |
|---|---|---|
| Distribution | Easy — once, in class | Awkward — issued individually |
| Blast radius of a leak | Everyone | Just that person |
| Usage tracking | Impossible | Possible |
| Cost of revoking | Everyone reconfigures | Just that person |

A real service always uses per-user keys with usage limits. We use a shared key
for convenience, so **be correspondingly careful.**

</details>

---
## 2주차 정리 · Week 2 summary

1. Git은 **무엇이 · 언제 · 왜** 바뀌었는지 기록한다. `add` → `commit` → `push`
2. 인증은 비밀번호가 아니라 **PAT**. 권한은 최소로
3. 반복 코드는 **모듈 하나**로 모으고, 런타임 초기화 후 되는지 확인한다
4. **프롬프트도 코드다.** 버전 관리하라
5. 키는 절대 커밋하지 않는다. **유출되면 폐기가 먼저다**

> **EN**
> 1. Git records **what, when and why** something changed: `add` → `commit` → `push`
> 2. Authentication is by **PAT**, not a password — least privilege
> 3. Gather repeated code into **one module**, then reset the runtime and verify
> 4. **A prompt is code** — version it
> 5. Never commit a key. **When one leaks, revoke first**

## 지금 확인할 것 · Check now

- [ ] 내 GitHub 저장소가 만들어졌다
- [ ] `src/kmu_llm.py` 가 올라가 있다
- [ ] `.gitignore` 에 `.env`, `*.key` 가 있다
- [ ] Colab Secrets 에 `KMU_API_KEY`, `GITHUB_TOKEN`, `GITHUB_USER` 가 있다
- [ ] 새 런타임에서 `wget` → `from kmu_llm import ask` 가 된다

> **EN**
> - [ ] My GitHub repository exists
> - [ ] `src/kmu_llm.py` has been pushed
> - [ ] `.gitignore` contains `.env` and `*.key`
> - [ ] Colab Secrets holds `KMU_API_KEY`, `GITHUB_TOKEN`, `GITHUB_USER`
> - [ ] In a fresh runtime, `wget` → `from kmu_llm import ask` works

## 다음 주 예고 · Coming next week

1주차 마지막에 모델이 숫자 대신 설명을 늘어놓아 파싱이 실패하는 것을 보았습니다.
3주차에는 그 문제를 정면으로 다룹니다.

- 프롬프트로 형식을 강제하는 법
- 그래도 깨질 때 파싱하고 검증하는 법
- 실패하면 오류를 되먹여 다시 요구하는 법

그 결과가 오늘 껍데기만 만들어 둔 **`ask_json()`** 입니다.
이 함수가 이후 모든 주차의 기반이 됩니다.

> **EN** — At the end of week 1 we watched a parse fail because the model wrote an
> explanation instead of a number. Week 3 takes that problem head on: how to force
> the format through the prompt, how to parse and validate when it breaks anyway,
> and how to feed the error back and ask again. The result is **`ask_json()`**,
> whose empty shell you wrote today — and which underpins every week that follows.

**수업 저장소** → https://github.com/zoomer1975-boop/ai_agent